# Brain Tumor MRI Classification using Deep Learning
### CNN + Transfer Learning + Grad-CAM

**Project goal:** classify brain MRI images into four dataset labels: `glioma`, `meningioma`, `pituitary`, and `notumor`.

**Important:** This is an educational image-classification project, not a medical diagnostic device. It must not be used to diagnose, rule out, or make treatment decisions about a person.

## Project roadmap

1. Prepare the dataset folders.
2. Install libraries and configure paths.
3. Inspect class counts and sample images (EDA).
4. Create training and validation sets; keep the test set untouched.
5. Handle imbalance with class weights and augmentation.
6. Train a custom CNN baseline.
7. Train an EfficientNetB0 transfer-learning model.
8. Fine-tune and compare results.
9. Evaluate with per-class precision, recall, F1 and a confusion matrix.
10. Explain predictions with Grad-CAM.
11. Save the model and class labels.
12. Optionally build a Streamlit demo.

### Dataset

Download the [Brain Tumor MRI Dataset from Kaggle](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset), extract it, and place the `Training` and `Testing` folders inside your project `Dataset` folder:

```text
Brain_Tumor_MRI_Project/
├── Dataset/
│   ├── Training/
│   │   ├── glioma/
│   │   ├── meningioma/
│   │   ├── notumor/
│   │   └── pituitary/
│   └── Testing/
│       ├── glioma/
│       ├── meningioma/
│       ├── notumor/
│       └── pituitary/
├── Notebook/
├── Models/
├── Results/
└── App/
```

## 1. Install the required libraries
Run this cell once. If TensorFlow was installed or upgraded, restart the Jupyter kernel.

## 2. Imports, seed and device check

In [ ]:
import os, json, random, pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_recall_fscore_support
)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## 3. Configure your Windows dataset path

Change `PROJECT_DIR` to the actual location of your project folder. Use a raw string (`r"..."`) for Windows paths.

Example: `r"C:\Users\Vinayak\Desktop\Brain_Tumor_MRI_Project"`

If your extracted folders are somewhere else, update the path accordingly.

In [ ]:
import os

# Main project directory
PROJECT_DIR = r"C:\Users\vinay\Brain_Tumor_Classification"

# Dataset paths (directly inside project folder)
TRAIN_DIR = os.path.join(PROJECT_DIR, "Training")
TEST_DIR = os.path.join(PROJECT_DIR, "Testing")

# Output folders
MODEL_DIR = os.path.join(PROJECT_DIR, "Models")
RESULT_DIR = os.path.join(PROJECT_DIR, "Results")
APP_DIR = os.path.join(PROJECT_DIR, "App")

# Verify paths
print("Training Folder Exists:", os.path.isdir(TRAIN_DIR))
print("Testing Folder Exists:", os.path.isdir(TEST_DIR))

# Create output folders
for folder in [MODEL_DIR, RESULT_DIR, APP_DIR]:
    os.makedirs(folder, exist_ok=True)

print("\nAll paths configured successfully!")

In [ ]:
# Check all training and testing classes

print("Training Classes:")
print(os.listdir(TRAIN_DIR))

print("\nTesting Classes:")
print(os.listdir(TEST_DIR))

## 4. Dataset audit and EDA

Before training, count valid image files in each class. Check that the same four class folders exist in Training and Testing. This dataset audit also helps identify class imbalance.

In [ ]:
VALID_EXTS = (".jpg", ".jpeg", ".png", ".bmp")

def count_images(folder):
    rows = []
    for class_name in sorted(os.listdir(folder)):
        class_path = os.path.join(folder, class_name)
        if os.path.isdir(class_path):
            count = sum(
                1 for name in os.listdir(class_path)
                if name.lower().endswith(VALID_EXTS)
            )
            rows.append({"class": class_name, "images": count})
    return pd.DataFrame(rows)

train_counts = count_images(TRAIN_DIR)
test_counts = count_images(TEST_DIR)

display(train_counts)
display(test_counts)

assert set(train_counts["class"]) == set(test_counts["class"]), (
    "Training and Testing class folder names do not match."
)

plt.figure(figsize=(9, 4))
sns.barplot(data=train_counts, x="class", y="images")
plt.title("Training images per class")
plt.xlabel("Class")
plt.ylabel("Number of images")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, "class_distribution.png"), dpi=150)
plt.show()

### Display sample MRI images

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 7))
for ax, class_name in zip(axes.flat, sorted(os.listdir(TRAIN_DIR))):
    class_path = os.path.join(TRAIN_DIR, class_name)
    files = [f for f in os.listdir(class_path) if f.lower().endswith(VALID_EXTS)]
    if files:
        image_path = os.path.join(class_path, files[0])
        img = tf.keras.utils.load_img(image_path, color_mode="rgb")
        ax.imshow(img)
        ax.set_title(class_name)
    ax.axis("off")
plt.suptitle("Example MRI images from the training set")
plt.tight_layout()
plt.show()

## 5. Create training, validation and test datasets

- 80% of the `Training` folder is used for model training.
- 20% of the `Training` folder is used for validation.
- The separate `Testing` folder is held back for final evaluation.

Do not tune the model by repeatedly checking test-set performance. That would leak information from the test set into model development.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.20,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    color_mode="rgb",
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.20,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    color_mode="rgb",
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    color_mode="rgb",
    shuffle=False
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print("Class order:", class_names)
print("Number of classes:", NUM_CLASSES)

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

## 6. Class imbalance handling

Class weights make mistakes on less frequent classes count more during training. Calculate weights using the training subset labels only.

This is one approach to imbalance; compare it with an unweighted baseline if you want to study its effect.

In [ ]:
train_labels = np.concatenate([labels.numpy() for _, labels in train_ds])
class_ids = np.arange(NUM_CLASSES)

weights = compute_class_weight(
    class_weight="balanced",
    classes=class_ids,
    y=train_labels
)
class_weights = {int(i): float(w) for i, w in zip(class_ids, weights)}
print("Class weights:", class_weights)

## 7. Data augmentation

Augmentation creates varied versions of training images. Keep transformations modest: aggressive rotations, flips or distortions may alter meaningful anatomical orientation or image characteristics. Augmentation layers are active during training and inactive during evaluation.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.04),
    tf.keras.layers.RandomZoom(0.08),
], name="data_augmentation")

## 8. Build a custom CNN baseline

This is the baseline model for comparison. It contains convolution, batch normalization, max pooling, dropout and dense classification layers.

In [ ]:
def build_custom_cnn(input_shape=(224, 224, 3), num_classes=4):
    inputs = tf.keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = tf.keras.layers.Rescaling(1./255)(x)

    for filters in [32, 64, 128]:
        x = tf.keras.layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation("relu")(x)
        x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.40)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    return tf.keras.Model(inputs, outputs, name="custom_cnn")

cnn_model = build_custom_cnn(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    num_classes=NUM_CLASSES
)
cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
cnn_model.summary()

## 9. Train the CNN baseline

Early stopping monitors validation loss and restores the best weights. ReduceLROnPlateau lowers the learning rate when validation performance stops improving.

In [ ]:
def make_callbacks(model_path):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            model_path, monitor="val_loss",
            save_best_only=True, verbose=1
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5,
            restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.2,
            patience=2, min_lr=1e-7, verbose=1
        )
    ]

cnn_path = os.path.join(MODEL_DIR, "best_custom_cnn.keras")
cnn_history = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    class_weight=class_weights,
    callbacks=make_callbacks(cnn_path)
)

### Plot baseline training curves

In [ ]:
def plot_history(history, title, save_path=None):
    h = pd.DataFrame(history.history)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(h["accuracy"], label="Train")
    axes[0].plot(h["val_accuracy"], label="Validation")
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(h["loss"], label="Train")
    axes[1].plot(h["val_loss"], label="Validation")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    fig.suptitle(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()

plot_history(
    cnn_history, "Custom CNN training history",
    os.path.join(RESULT_DIR, "cnn_training_curves.png")
)

## 10. Transfer learning with EfficientNetB0

EfficientNetB0 is pretrained on ImageNet. We freeze its feature extractor first and train a new classification head for the MRI dataset.

Keras EfficientNet models include their own input rescaling/preprocessing. Therefore, pass image tensors in the 0–255 range to this model; do not divide by 255 in this branch.

In [ ]:
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.35)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)

efficientnet_model = tf.keras.Model(inputs, outputs, name="efficientnetb0_mri")
efficientnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
efficientnet_model.summary()

## 11. Train EfficientNetB0

In [ ]:
eff_path = os.path.join(MODEL_DIR, "best_efficientnetb0.keras")
eff_history = efficientnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    class_weight=class_weights,
    callbacks=make_callbacks(eff_path)
)

plot_history(
    eff_history, "EfficientNetB0 training history",
    os.path.join(RESULT_DIR, "efficientnet_training_curves.png")
)

## 12. Fine-tune EfficientNetB0

After training the classifier head, unfreeze only a small number of top backbone layers. Use a very small learning rate. Keep BatchNormalization layers frozen to reduce instability.

If validation loss becomes worse, stop and use the best checkpoint from the frozen-backbone stage.

In [ ]:
base_model.trainable = True

# Fine-tune only the last 30 layers of the backbone
for layer in base_model.layers[:-30]:
    layer.trainable = False

for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

efficientnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

fine_path = os.path.join(MODEL_DIR, "best_efficientnetb0_finetuned.keras")
fine_history = efficientnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    class_weight=class_weights,
    callbacks=make_callbacks(fine_path)
)

plot_history(
    fine_history, "EfficientNetB0 fine-tuning history",
    os.path.join(RESULT_DIR, "efficientnet_finetuning_curves.png")
)

## 13. Evaluate models on the untouched test set

Use the best saved checkpoint. Accuracy alone is not sufficient: inspect per-class recall and macro-F1, especially when class counts differ.

In [ ]:
def evaluate_model(model, dataset, labels, model_name):
    y_true = np.concatenate([y.numpy() for _, y in dataset])
    probabilities = model.predict(dataset, verbose=1)
    y_pred = np.argmax(probabilities, axis=1)

    print(f"\n===== {model_name} =====")
    print(classification_report(
        y_true, y_pred, target_names=labels,
        digits=4, zero_division=0
    ))

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    accuracy = accuracy_score(y_true, y_pred)

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 6))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=labels, yticklabels=labels
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"{model_name} – Confusion Matrix")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, f"{model_name}_confusion_matrix.png"), dpi=150)
    plt.show()

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Macro Precision": macro_p,
        "Macro Recall": macro_r,
        "Macro F1": macro_f1
    }

best_cnn = tf.keras.models.load_model(cnn_path)
best_eff = tf.keras.models.load_model(fine_path)

results = []
results.append(evaluate_model(best_cnn, test_ds, class_names, "Custom_CNN"))
results.append(evaluate_model(best_eff, test_ds, class_names, "EfficientNetB0"))

comparison_df = pd.DataFrame(results).sort_values("Macro F1", ascending=False)
display(comparison_df)
comparison_df.to_csv(os.path.join(RESULT_DIR, "model_comparison.csv"), index=False)

In [ ]:
!pip install streamlit


In [ ]:
cd C:\Users\vinay\Brain_Tumor_Classification

## 14. Grad-CAM explainability for EfficientNetB0

Grad-CAM creates a heatmap indicating image regions that influenced a selected class score. It is a visualization aid, not proof that the model uses medically meaningful evidence.

This helper locates the final convolutional layer automatically. If the layer lookup fails for your TensorFlow/Keras version, inspect `best_eff.layers` and set `LAST_CONV_LAYER` to the final 4-D convolutional feature layer inside the backbone.

In [ ]:
# Inspect candidate convolutional layers in the EfficientNet backbone
for layer in base_model.layers[-15:]:
    print(layer.name, layer.output.shape)

In [ ]:
def find_last_conv_layer(backbone):
    for layer in reversed(backbone.layers):
        try:
            shape = layer.output.shape
            if len(shape) == 4:
                return layer.name
        except Exception:
            pass
    raise ValueError("Could not find a 4D feature layer.")

LAST_CONV_LAYER = find_last_conv_layer(base_model)
print("Selected Grad-CAM layer:", LAST_CONV_LAYER)

In [ ]:
def make_gradcam_heatmap(image_batch, full_model, backbone, last_conv_layer_name, class_index=None):
    # Create one backbone graph that returns both the selected feature map and final backbone output.
    conv_layer = backbone.get_layer(last_conv_layer_name)
    feature_model = tf.keras.Model(
        inputs=backbone.input,
        outputs=[conv_layer.output, backbone.output]
    )

    augmentation_layer = full_model.get_layer("data_augmentation")
    # Layers after the nested backbone form the classifier head.
    backbone_position = next(
        i for i, layer in enumerate(full_model.layers)
        if layer.name == backbone.name
    )
    head_layers = full_model.layers[backbone_position + 1:]

    with tf.GradientTape() as tape:
        augmented = augmentation_layer(image_batch, training=False)
        conv_outputs, backbone_features = feature_model(augmented, training=False)

        x = backbone_features
        for layer in head_layers:
            x = layer(x, training=False)
        predictions = x

        if class_index is None:
            class_index = tf.argmax(predictions[0])
        class_score = predictions[:, class_index]

    grads = tape.gradient(class_score, conv_outputs)
    if grads is None:
        raise RuntimeError("Gradients are unavailable. Try another convolutional feature layer.")

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + tf.keras.backend.epsilon())
    return heatmap.numpy(), int(class_index.numpy()), predictions.numpy()[0]

### Run Grad-CAM on one test image

Select an image path from one of the test class folders. The code shows the MRI and heatmap overlay.

In [ ]:
# CHANGE this to a real image path from your Testing folder
SAMPLE_IMAGE_PATH = r"C:\path\to\Dataset\Testing\glioma\example.jpg"

def show_gradcam(image_path):
    image = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE, color_mode="rgb")
    array = tf.keras.utils.img_to_array(image)
    batch = tf.expand_dims(array, axis=0)

    heatmap, predicted_index, probs = make_gradcam_heatmap(
        batch, best_eff, base_model, LAST_CONV_LAYER
    )

    heatmap_resized = tf.image.resize(
        heatmap[..., np.newaxis], IMG_SIZE
    ).numpy().squeeze()

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.imshow(image)
    plt.title("Original MRI")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(heatmap_resized, cmap="jet")
    plt.title("Grad-CAM heatmap")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(image)
    plt.imshow(heatmap_resized, cmap="jet", alpha=0.35)
    plt.title("Overlay")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

    print("Predicted dataset label:", class_names[predicted_index])
    for name, score in sorted(zip(class_names, probs), key=lambda x: x[1], reverse=True):
        print(f"{name}: {score*100:.2f}%")

# Uncomment after setting a valid image path:
# show_gradcam(SAMPLE_IMAGE_PATH)

## 15. Predict one new MRI image

The function below loads a single image, resizes it to the model input size and returns the model's class scores. These are model scores, not calibrated clinical probabilities.

In [ ]:
def predict_mri(image_path, model=best_eff):
    image = tf.keras.utils.load_img(
        image_path, target_size=IMG_SIZE, color_mode="rgb"
    )
    array = tf.keras.utils.img_to_array(image)
    batch = np.expand_dims(array, axis=0)  # Keep 0–255 for EfficientNet

    scores = model.predict(batch, verbose=0)[0]
    order = np.argsort(scores)[::-1]

    plt.figure(figsize=(5, 5))
    plt.imshow(image)
    plt.axis("off")
    plt.title("Uploaded MRI image")
    plt.show()

    print("Predicted dataset label:", class_names[int(order[0])])
    print("Model scores (not diagnostic probabilities):")
    for idx in order:
        print(f"{class_names[idx]}: {scores[idx]*100:.2f}%")

    return class_names[int(order[0])], scores

# Example:
# predict_mri(r"C:\path\to\your\MRI_image.jpg")

## 16. Save the final model and class labels

Always save the class order alongside the model. It is required to interpret output indices correctly when deploying.

In [ ]:
FINAL_MODEL_PATH = os.path.join(MODEL_DIR, "brain_tumor_efficientnet.keras")
best_eff.save(FINAL_MODEL_PATH)

with open(os.path.join(MODEL_DIR, "class_names.json"), "w", encoding="utf-8") as file:
    json.dump(class_names, file, indent=2)

print("Saved model:", FINAL_MODEL_PATH)
print("Saved labels:", os.path.join(MODEL_DIR, "class_names.json"))

## 17. Optional: create a Streamlit web application

Run this cell to create `App/app.py`. Then open a Windows terminal in the project folder and run:

```bash
streamlit run App/app.py
```

The app displays a model prediction and score ranking. Keep the non-diagnostic warning visible.

In [ ]:
app_code = r'''
import streamlit as st
import tensorflow as tf
import numpy as np
import json
from PIL import Image

st.set_page_config(page_title="Brain MRI Classifier", page_icon="🧠")
st.title("Brain MRI Image Classification")
st.warning(
    "Educational demonstration only. This tool is not a medical device "
    "and cannot diagnose or rule out a brain tumor. Consult a qualified clinician."
)

MODEL_PATH = "Models/brain_tumor_efficientnet.keras"
LABEL_PATH = "Models/class_names.json"

@st.cache_resource
def load_artifacts():
    model = tf.keras.models.load_model(MODEL_PATH)
    with open(LABEL_PATH, "r", encoding="utf-8") as f:
        labels = json.load(f)
    return model, labels

uploaded = st.file_uploader("Upload an MRI image", type=["jpg", "jpeg", "png"])
if uploaded is not None:
    image = Image.open(uploaded).convert("RGB").resize((224, 224))
    st.image(image, caption="Uploaded image", use_container_width=True)

    if st.button("Run educational classification demo"):
        model, labels = load_artifacts()
        arr = np.asarray(image, dtype=np.float32)[None, ...]
        scores = model.predict(arr, verbose=0)[0]
        ranking = np.argsort(scores)[::-1]

        st.subheader("Model output")
        st.write("Top dataset label:", labels[int(ranking[0])])
        st.caption("Scores are not calibrated clinical probabilities.")
        for idx in ranking:
            st.write(f"{labels[idx]}: {scores[idx]*100:.2f}%")
'''
app_path = os.path.join(APP_DIR, "app.py")
with open(app_path, "w", encoding="utf-8") as f:
    f.write(app_code)

print("Created:", app_path)

## 18. How to improve this project scientifically

1. Compare the custom CNN against EfficientNetB0 using the same data split and metrics.
2. Add VGG16 and ResNet50 only after the first two models run correctly. Record training time and model size as well as scores.
3. Compare class weights against no class weights; report macro-F1 and per-class recall.
4. Check for duplicate and near-duplicate images across training, validation and testing. If patient IDs are available, split by patient, not by image.
5. Check for dataset source or acquisition artifacts that could let the model learn shortcuts.
6. Use restrained augmentation and document every preprocessing step.
7. Evaluate robustness to brightness, contrast and image-quality changes.
8. Do not claim clinical accuracy based only on this public dataset. External validation and expert review would be needed for clinical use.

### Suggested report sections

- Introduction, problem statement, aim and objectives
- Literature survey
- Dataset description and class distribution
- Preprocessing and augmentation
- CNN architecture and transfer learning
- Training setup and hyperparameters
- Results: accuracy, macro precision, macro recall, macro-F1, confusion matrix
- Grad-CAM examples and limitations
- Conclusion and future scope

### Troubleshooting

- **Training folder not found:** verify `PROJECT_DIR` and that `Dataset/Training` exists.
- **Out of memory:** reduce `BATCH_SIZE` from 32 to 16 or 8.
- **TensorFlow cannot find GPU:** CPU training is still possible; check TensorFlow and GPU/CUDA compatibility before troubleshooting.
- **Grad-CAM layer error:** inspect the printed EfficientNet layer names and choose a final 4-D feature layer.
- **Low validation performance:** inspect labels, class counts, duplicates, preprocessing and learning curves before changing the architecture.